# Proyecto Final — Etapa 2: Clasificación de Textos por Década (Mark 5)

**Mejoras sobre Mark 4:**
- **FULL_TRAIN=True** — 100% de los datos de entrenamiento
- **LR=3e-5**, warmup=10% — más agresivo
- **Ordinal CE loss**
- **Ensemble 4 semillas** (mejorado desde 2)
- **Modelo:** `PlanTL-GOB-ES/roberta-base-bne` — Apache 2.0 — https://huggingface.co/PlanTL-GOB-ES/roberta-base-bne

## 1. Instalación

In [1]:
!pip install transformers accelerate -q

## 2. Librerías

In [2]:
import os, re, random
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.optim import AdamW

print(f"PyTorch: {torch.__version__}")

PyTorch: 2.10.0+cu128


## 3. Configuración

In [3]:
import os # Added this line
import torch # Added this line
from contextlib import nullcontext # Added this line

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB  = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_KAGGLE:
    DATA_DIR = "/kaggle/input/machine-learning-project"
    OUT_DIR  = "/kaggle/working"
elif IS_COLAB:
    DATA_DIR = "/content"
    OUT_DIR  = "/content"
else:
    DATA_DIR = "../../data"
    OUT_DIR  = "../submissions"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
EVAL_PATH  = os.path.join(DATA_DIR, "eval.csv")
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME   = "roberta-base"
MAX_LEN      = 128
STRIDE       = 64
BATCH_SIZE   = 32
GRAD_ACCUM   = 1
LABEL_SMOOTH = 0.05
WARMUP_FRAC  = 0.10
SEEDS        = [42, 2024, 123, 456]
FULL_TRAIN   = True    # True = 100% datos  |  False = 90/10 split con early stop
PATIENCE     = 3       # solo aplica si FULL_TRAIN=False
EPOCHS       = 6 if FULL_TRAIN else 10
LR           = 3e-5
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP      = torch.cuda.is_available()

if USE_AMP:
    try:
        _scaler   = torch.amp.GradScaler("cuda")
        _autocast = lambda: torch.amp.autocast("cuda")
    except AttributeError:
        _scaler   = torch.cuda.GradScaler()
        _autocast = torch.cuda.amp.autocast
else:
    _scaler, _autocast = None, nullcontext

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

print(f"Entorno: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")
print(f"Device:  {DEVICE}  |  FULL_TRAIN={FULL_TRAIN}  |  EPOCHS={EPOCHS}  |  LR={LR}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Entorno: Colab
Device:  cuda  |  FULL_TRAIN=True  |  EPOCHS=6  |  LR=3e-05
GPU: Tesla T4  |  15.6 GB


## 4. Datos

In [4]:
def clean(text):
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", str(text))
    return re.sub(r" {2,}", " ", text).strip()

train_df = pd.read_csv(TRAIN_PATH)
eval_df  = pd.read_csv(EVAL_PATH)
train_df["text"] = train_df["text"].map(clean)
eval_df["text"]  = eval_df["text"].map(clean)

le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["decade"])
NUM_CLASSES = len(le.classes_)

if FULL_TRAIN:
    train_split = train_df
    val_split   = None
else:
    train_split, val_split = train_test_split(
        train_df, test_size=0.1, random_state=42, stratify=train_df["label"]
    )

print(f"Train: {len(train_split)}  |  Val: {len(val_split) if val_split is not None else 0}  |  Eval: {len(eval_df)}  |  Clases: {NUM_CLASSES}")

Train: 31403  |  Val: 0  |  Eval: 3490  |  Clases: 39


## 5. Dataset y DataLoader

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class DecadeDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True) if labels is not None else None

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(str(self.texts[idx]), max_length=MAX_LEN,
                        padding="max_length", truncation=True, return_tensors="pt")
        item = {"input_ids": enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze()}
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item


def make_loaders(train_s, val_s=None):
    tr = DataLoader(DecadeDataset(train_s["text"], train_s["label"]),
                    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=USE_AMP)
    va = DataLoader(DecadeDataset(val_s["text"], val_s["label"]),
                    batch_size=BATCH_SIZE * 2, num_workers=2, pin_memory=USE_AMP) if val_s is not None else None
    return tr, va


print("Tokenizador listo")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizador listo


## 6. Pérdida ordinal + funciones de entrenamiento

**Ordinal CE loss:** cross-entropy + penalización proporcional a la distancia entre décadas.

In [6]:
ORDINAL_W = torch.zeros(NUM_CLASSES, NUM_CLASSES, device=DEVICE)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ORDINAL_W[i, j] = abs(i - j)
ORDINAL_W = ORDINAL_W / ORDINAL_W.max()

ALPHA = 0.3


def ordinal_ce_loss(logits, labels):
    ce   = F.cross_entropy(logits, labels, label_smoothing=LABEL_SMOOTH)
    probs = F.softmax(logits, dim=-1)
    pen  = (probs * ORDINAL_W[labels]).sum(-1).mean()
    return (1 - ALPHA) * ce + ALPHA * pen


def train_one(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(loader):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        with _autocast():
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = ordinal_ce_loss(logits, lbls) / GRAD_ACCUM
        if _scaler: _scaler.scale(loss).backward()
        else:       loss.backward()
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            if _scaler: _scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if _scaler: _scaler.step(optimizer); _scaler.update()
            else:       optimizer.step()
            scheduler.step(); optimizer.zero_grad()
        total_loss += loss.item() * GRAD_ACCUM
        correct    += (logits.detach().argmax(-1) == lbls).sum().item()
        n          += len(lbls)
    return total_loss / len(loader), correct / n


@torch.no_grad()
def eval_one(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        with _autocast():
            logits = model(input_ids=ids, attention_mask=mask).logits
        total_loss += F.cross_entropy(logits, lbls).item()
        correct    += (logits.argmax(-1) == lbls).sum().item()
        n          += len(lbls)
    return total_loss / len(loader), correct / n


def fit(train_s, val_s, seed, tag):
    set_seed(seed)
    tr_loader, va_loader = make_loaders(train_s, val_s)

    m = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
    ).to(DEVICE)

    opt   = AdamW(m.parameters(), lr=LR, weight_decay=0.01)
    steps = (len(tr_loader) // GRAD_ACCUM) * EPOCHS
    sch   = get_cosine_schedule_with_warmup(opt, max(1, int(WARMUP_FRAC * steps)), steps)

    best_acc, patience_cnt = 0.0, 0
    ckpt = os.path.join(OUT_DIR, f"ckpt_{tag}.pt")

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one(m, tr_loader, opt, sch)
        if va_loader:
            va_loss, va_acc = eval_one(m, va_loader)
            print(f"[{tag}] Epoch {epoch}/{EPOCHS}  tr_acc={tr_acc:.4f}  va_acc={va_acc:.4f}")
            if va_acc > best_acc:
                best_acc, patience_cnt = va_acc, 0
                torch.save(m.state_dict(), ckpt)
            else:
                patience_cnt += 1
                if patience_cnt >= PATIENCE:
                    print(f"  Early stop epoch {epoch}"); break
        else:
            print(f"[{tag}] Epoch {epoch}/{EPOCHS}  tr_acc={tr_acc:.4f}")
            torch.save(m.state_dict(), ckpt)

    if va_loader:
        print(f"[{tag}] Mejor val_acc: {best_acc:.4f}")
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return m


print("Funciones listas")

Funciones listas


## 7. Entrenamiento

In [ ]:
models = []
for seed in SEEDS:
    print(f"\n{'='*50}\nEntrenando seed={seed}  ({'100% datos' if FULL_TRAIN else 'val split'})\n{'='*50}")
    m = fit(train_split, val_split, seed=seed, tag=f"seed{seed}")
    models.append(m)

print(f"\nEnsemble de {len(models)} modelos listo")


Entrenando seed=42  (100% datos)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[seed42] Epoch 1/6  tr_acc=0.0998
[seed42] Epoch 2/6  tr_acc=0.1799
[seed42] Epoch 3/6  tr_acc=0.2289
[seed42] Epoch 4/6  tr_acc=0.2764
[seed42] Epoch 5/6  tr_acc=0.3197
[seed42] Epoch 6/6  tr_acc=0.3443

Entrenando seed=2024  (100% datos)


...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[seed123] Epoch 1/6  tr_acc=0.1027
[seed123] Epoch 2/6  tr_acc=0.1767
[seed123] Epoch 3/6  tr_acc=0.2265
[seed123] Epoch 4/6  tr_acc=0.2796
[seed123] Epoch 5/6  tr_acc=0.3211
[seed123] Epoch 6/6  tr_acc=0.3475

Entrenando seed=456  (100% datos)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[seed456] Epoch 1/6  tr_acc=0.1015
[seed456] Epoch 2/6  tr_acc=0.1804
[seed456] Epoch 3/6  tr_acc=0.2313


## 8. Inferencia con sliding window + ensemble y Envío

In [ ]:
from transformers import AutoTokenizer # Added this line
import torch # Added this line
import torch.nn.functional as F # Added this line

# The MODEL_NAME variable is expected to be defined in cell-6.
# If cell-6 has not been executed successfully, this line will still fail.
# It's crucial to run cell-6 and other preceding cells first.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) # Added this line

CLS_ID = tokenizer.cls_token_id or tokenizer.bos_token_id
SEP_ID = tokenizer.sep_token_id or tokenizer.eos_token_id
PAD_ID = tokenizer.pad_token_id
CHUNK  = MAX_LEN - 2


@torch.no_grad()
def get_probs(model, texts):
    all_probs = []
    model.eval()
    for text in texts:
        tids = tokenizer.encode(str(text), add_special_tokens=False)
        wins_ids, wins_mask = [], []
        pos = 0
        while pos < max(len(tids), 1):
            chunk   = tids[pos: pos + CHUNK]
            seq     = [CLS_ID] + chunk + [SEP_ID]
            pad_len = MAX_LEN - len(seq)
            wins_ids.append(seq + [PAD_ID] * pad_len)
            wins_mask.append([1] * len(seq) + [0] * pad_len)
            pos += STRIDE
            if pos >= len(tids): break
        t_ids  = torch.tensor(wins_ids,  dtype=torch.long).to(DEVICE)
        t_mask = torch.tensor(wins_mask, dtype=torch.long).to(DEVICE)
        with _autocast():
            logits = model(input_ids=t_ids, attention_mask=t_mask).logits
        all_probs.append(F.softmax(logits.mean(0), dim=-1).cpu())
    return torch.stack(all_probs)


print("Generando predicciones del ensemble...")
texts_eval = eval_df["text"].tolist()

ensemble_probs = sum(get_probs(m, texts_eval) for m in models) / len(models)
preds = ensemble_probs.argmax(-1).numpy()

submission = pd.DataFrame({"id": eval_df["id"], "answer": le.inverse_transform(preds)})
sub_path   = os.path.join(OUT_DIR, "submission_mark5.csv")
submission.to_csv(sub_path, index=False)
print(f"Submission guardado: {sub_path}")
print(f"Distribución:\n{submission['answer'].value_counts().sort_index()}")
submission.head()